In [23]:
import os
import time
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


from datasets import load_dataset

In [24]:
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1066 entries, 0 to 1065
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    1066 non-null   object
 1   label   1066 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 16.8+ KB


In [25]:
test['label'] = test['label'].apply(lambda x: 'positive' if x == 1 else 'negative')

labels = test['label'].unique()

test

,text,label
0,lovingly photographed in the manner of a golde...,positive
1,consistently clever and suspenseful .,positive
2,"it's like a "" big chill "" reunion of the baade...",positive
3,the story gives ample opportunity for large-sc...,positive
4,"red dragon "" never cuts corners .",positive
...,...,...
1061,a terrible movie that some people will neverth...,negative
1062,there are many definitions of 'time waster' bu...,negative
1063,"as it stands , crocodile hunter has the hurrie...",negative
1064,the thing looks like a made-for-home-video qui...,negative


In [26]:
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [27]:
def classify(text, labels):
    start_time = time.time()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        store=True,
        messages = [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis of movie reviews. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
        ],
    )

    request_time = time.time() - start_time
    completion = response.choices[0].message.content.lower()
    completion_tokens = response.usage.completion_tokens
    prompt_tokens = response.usage.prompt_tokens
    total_tokens = response.usage.total_tokens

    return completion, request_time, completion_tokens, prompt_tokens, total_tokens

def post_process(text):
    if 'positive' in text:
        return 'positive'
    elif 'negative' in text:
        return 'negative'
    else:
        return 'error'

In [29]:
pred_df = test.copy() 

for index, row in pred_df.iterrows():
    try:
        text = row['text']
        completion, request_time, completion_tokens, prompt_tokens, total_tokens = classify(text, labels)
        pred_df.at[index, 'prediction'] = completion
        pred_df.at[index, 'request_time'] = request_time
        pred_df.at[index, 'completion_tokens'] = completion_tokens
        pred_df.at[index, 'prompt_tokens'] = prompt_tokens
        pred_df.at[index, 'total_tokens'] = total_tokens

    except Exception as e:
        # Save the current state of the DataFrame to a file before breaking out or retrying.
        pred_df.to_csv("results/partial_openai_ZS_binary2.csv", index=False)
        print(f"An error occurred at index {index}: {e}. Partial results saved.")
        # Optionally, you can break out of the loop or continue based on your needs.
        break

pred_df['prediction_post_processed'] = pred_df['prediction'].apply(post_process)
pred_df.to_csv("results/openai_ZS_binary2.csv", index=False)

pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,lovingly photographed in the manner of a golde...,positive,positive,0.956203,2.0,113.0,115.0,positive
1,consistently clever and suspenseful .,positive,positive,0.633795,2.0,94.0,96.0,positive
2,"it's like a "" big chill "" reunion of the baade...",positive,negative,0.592738,2.0,117.0,119.0,negative
3,the story gives ample opportunity for large-sc...,positive,positive,0.520349,2.0,112.0,114.0,positive
4,"red dragon "" never cuts corners .",positive,positive,0.563910,2.0,95.0,97.0,positive
...,...,...,...,...,...,...,...,...
1061,a terrible movie that some people will neverth...,negative,negative,0.402339,2.0,99.0,101.0,negative
1062,there are many definitions of 'time waster' bu...,negative,negative,0.471366,2.0,108.0,110.0,negative
1063,"as it stands , crocodile hunter has the hurrie...",negative,negative,0.496069,2.0,134.0,136.0,negative
1064,the thing looks like a made-for-home-video qui...,negative,negative,0.540304,2.0,100.0,102.0,negative


In [30]:
pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,lovingly photographed in the manner of a golde...,positive,positive,0.956203,2.0,113.0,115.0,positive
1,consistently clever and suspenseful .,positive,positive,0.633795,2.0,94.0,96.0,positive
2,"it's like a "" big chill "" reunion of the baade...",positive,negative,0.592738,2.0,117.0,119.0,negative
3,the story gives ample opportunity for large-sc...,positive,positive,0.520349,2.0,112.0,114.0,positive
4,"red dragon "" never cuts corners .",positive,positive,0.563910,2.0,95.0,97.0,positive
...,...,...,...,...,...,...,...,...
1061,a terrible movie that some people will neverth...,negative,negative,0.402339,2.0,99.0,101.0,negative
1062,there are many definitions of 'time waster' bu...,negative,negative,0.471366,2.0,108.0,110.0,negative
1063,"as it stands , crocodile hunter has the hurrie...",negative,negative,0.496069,2.0,134.0,136.0,negative
1064,the thing looks like a made-for-home-video qui...,negative,negative,0.540304,2.0,100.0,102.0,negative


In [31]:
y_pred = pred_df['prediction_post_processed']
y_true = pred_df['label']

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.894934
F1 score: 0.895296
Precision: 0.904017
Recall: 0.894934


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [32]:
# get average response time, vram usage and ram usage
request_time_avg = pred_df['request_time'].mean()
completion_tokens_avg = pred_df['completion_tokens'].mean()
prompt_tokens_avg = pred_df['prompt_tokens'].mean()
total_tokens_avg = pred_df['total_tokens'].mean()

print(f'Average response time: {request_time_avg}')
print(f'Average completion tokens: {completion_tokens_avg}')
print(f'Average prompt tokens: {prompt_tokens_avg}')
print(f'Average total tokens: {total_tokens_avg}')

Average response time: 0.7937980326657894
Average completion tokens: 2.0
Average prompt tokens: 112.17823639774859
Average total tokens: 114.17823639774859


In [37]:
input_token_price = 0.15/1_000_000
output_token_price = 0.6/1_000_000

def count_tokens(text, model="gpt-4o-mini"):
    try:
        # Try to get the encoding for the given model
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        # If the model isn't recognized, fall back to a default encoding
        encoding = tiktoken.get_encoding("cl100k_base")
    
    tokens = encoding.encode(text)
    return len(tokens)

# Calculate the cost of the requests
total_cost = 0
for index, row in pred_df.iterrows():
    completion_tokens = row['completion_tokens']
    prompt_tokens = row['prompt_tokens']
    cost  = completion_tokens * output_token_price + prompt_tokens * input_token_price
    total_cost += cost

print(f'Total cost: USD {total_cost}')

Total cost: USD 0.01921649999999998


In [38]:
with open('results/openai_ZS_binary2.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {request_time_avg}\n')
    f.write(f'Average completion tokens: {completion_tokens_avg}\n')
    f.write(f'Average prompt tokens: {prompt_tokens_avg}\n')
    f.write(f'Average total tokens: {total_tokens_avg}\n')
    f.write(f'Total cost: USD {total_cost}\n')